# Chapter 7: Working with Text Data

## Chapter Summary
Text must be converted into numeric features. This chapter builds bag-of-words and TF-IDF representations, compares stop-word and n-gram choices, inspects learned coefficients, and demonstrates topic modeling with Latent Dirichlet Allocation (LDA).

## Theoretical Deep-Dive

A document-term matrix records token counts. TF-IDF downweights terms common across many documents: $tfidf(t,d)=tf(t,d)\log((N+1)/(df(t)+1))$. Logistic regression then learns which terms push a prediction toward each class.

n-grams preserve short word order at the cost of a larger, sparser vocabulary. Tokenization defines the model's evidence, so casing, punctuation, stemming, and domain-specific stop words are modeling choices rather than neutral cleaning.

LDA treats each document as a mixture of latent topics and each topic as a distribution over words. Topics require human interpretation and stability checks.

**Book coverage:** sentiment analysis, bag-of-words, stop words, TF-IDF, coefficients, n-grams, tokenization, and LDA (Chapter 7, printed pp. 323-355).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

positive = [
    "a thoughtful and moving film", "excellent acting and a wonderful story", "funny warm and beautifully directed",
    "the performances were brilliant", "an exciting film with a satisfying ending", "smart writing and memorable characters",
    "I loved every minute of this movie", "a charming and uplifting experience", "visually stunning and emotionally rich",
    "the plot was engaging and original", "great pacing with strong performances", "an impressive and enjoyable production",
]
negative = [
    "a dull and predictable film", "terrible acting and a weak story", "boring cold and poorly directed",
    "the performances were painful", "a confusing film with a disappointing ending", "lazy writing and forgettable characters",
    "I regretted watching this movie", "a tedious and frustrating experience", "visually messy and emotionally empty",
    "the plot was repetitive and unoriginal", "awful pacing with flat performances", "a disappointing and unpleasant production",
]
texts = np.array(positive + negative); labels = np.array([1] * len(positive) + [0] * len(negative))
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.33, stratify=labels, random_state=42)
vectorizer = CountVectorizer()
counts = vectorizer.fit_transform(X_train)
print("Document-term shape:", counts.shape)
print("Vocabulary sample:", vectorizer.get_feature_names_out()[:15])

Document-term shape: (16, 55)
Vocabulary sample: ['acting' 'an' 'and' 'awful' 'boring' 'brilliant' 'characters' 'cold'
 'directed' 'disappointing' 'dull' 'emotionally' 'empty' 'ending'
 'engaging']


In [2]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
representations = {
    "counts": CountVectorizer(),
    "counts + stopwords": CountVectorizer(stop_words="english"),
    "tfidf": TfidfVectorizer(stop_words="english"),
    "tfidf + 1-2 grams": TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1),
}
rows = []
for name, transformer in representations.items():
    pipe = make_pipeline(transformer, LogisticRegression(max_iter=2000))
    score = cross_val_score(pipe, texts, labels, cv=cv, scoring="accuracy")
    pipe.fit(X_train, y_train)
    rows.append((name, score.mean(), pipe.score(X_test, y_test), len(pipe.named_steps[transformer.__class__.__name__.lower()].get_feature_names_out())))
display(pd.DataFrame(rows, columns=["representation", "cv_accuracy", "test_accuracy", "vocabulary_size"]).round(3))

,representation,cv_accuracy,test_accuracy,vocabulary_size
0,counts,0.375,0.125,55
1,counts + stopwords,0.375,0.125,45
2,tfidf,0.375,0.125,45
3,tfidf + 1-2 grams,0.375,0.125,84


In [3]:
sentiment = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=2000))
sentiment.fit(texts, labels)
tfidf = sentiment.named_steps["tfidfvectorizer"]
clf = sentiment.named_steps["logisticregression"]
terms = tfidf.get_feature_names_out()
weights = clf.coef_[0]
top_negative = terms[np.argsort(weights)[:8]]
top_positive = terms[np.argsort(weights)[-8:][::-1]]
print("Strong negative evidence:", top_negative)
print("Strong positive evidence:", top_positive)
for review, probability in zip(["wonderful acting and smart writing", "boring plot and terrible ending"], sentiment.predict_proba(["wonderful acting and smart writing", "boring plot and terrible ending"])[:, 1]):
    print(f"P(positive)={probability:.3f} | {review}")

Strong negative evidence: ['disappointing' 'painful' 'were painful' 'dull' 'predictable'
 'predictable film' 'dull and' 'and predictable']
Strong positive evidence: ['an' 'brilliant' 'were brilliant' 'thoughtful and' 'moving film'
 'and moving' 'thoughtful' 'moving']
P(positive)=0.550 | wonderful acting and smart writing
P(positive)=0.459 | boring plot and terrible ending


In [4]:
from sklearn.decomposition import LatentDirichletAllocation

topic_documents = [
    "team won the football match with two goals", "coach prepared the team for the league game", "players trained for the championship match",
    "new software improves computer security", "python code trains a machine learning model", "database servers process cloud applications",
    "fresh vegetables make a healthy meal", "the chef prepared pasta with tomato sauce", "baking bread requires flour and yeast",
    "election voters debated government policy", "parliament approved a new public law", "the minister announced an economic policy",
]
topic_vectorizer = CountVectorizer(stop_words="english", min_df=1)
topic_matrix = topic_vectorizer.fit_transform(topic_documents)
lda = LatentDirichletAllocation(n_components=4, max_iter=40, learning_method="batch", random_state=42)
document_topics = lda.fit_transform(topic_matrix)
feature_names = topic_vectorizer.get_feature_names_out()
for index, topic in enumerate(lda.components_):
    print(f"Topic {index}:", ", ".join(feature_names[np.argsort(topic)[-6:][::-1]]))
print("Dominant topic per document:", document_topics.argmax(axis=1))

Topic 0: yeast, requires, baking, flour, bread, public
Topic 1: prepared, tomato, sauce, pasta, chef, team
Topic 2: match, team, new, servers, applications, process
Topic 3: policy, trains, python, learning, machine, model
Dominant topic per document: [2 0 2 2 3 2 3 1 0 0 0 3]


## Key Takeaways
- Text models learn from a representation, not raw language.
- Bag-of-words discards order; n-grams recover short local context.
- TF-IDF reduces the influence of globally common terms.
- Coefficient inspection provides a useful model audit for linear text classifiers.
- Topic labels are human interpretations of word distributions, not objective ground truth.